# Regression Models for Network Flow Duration Prediction

This notebook implements 10 regression models to predict `flow_duration` in IoT network traffic. All models use the same preprocessing decisions and train/test split from the regression preprocessing stage for fair comparison.

**Models Implemented:**
- Tree-based: Random Forest, Gradient Boosting, Decision Tree
- Linear: Linear Regression, Ridge, Lasso, ElasticNet  
- Other: SVR, KNN (with scaling analysis)

**Target Variable:** `flow_duration` (continuous, in seconds)
**Evaluation Metrics:** R², RMSE, MAE

## Load Data and Split

We load the cleaned dataset and use the saved train/test indices to ensure reproducibility across all models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

# Load cleaned dataset
df = pd.read_csv("../data/RT_IOT2022_cleaned.csv")
print("Dataset shape:", df.shape)

In [ ]:
# Load saved split indices
split_data = np.load("regression_split_indices.npz")
train_indices = split_data['train_indices']
test_indices = split_data['test_indices']

print(f"Train indices: {len(train_indices)}")
print(f"Test indices: {len(test_indices)}")

In [ ]:
# Apply the same feature selection as preprocessing notebook
target = "flow_duration"

# Columns to drop (same as preprocessing)
drop_cols = []
timing_cols = [col for col in df.columns if 'iat' in col.lower()]
drop_cols.extend(timing_cols)
active_cols = [col for col in df.columns if col.startswith('active.')]
drop_cols.extend(active_cols)
idle_cols = [col for col in df.columns if col.startswith('idle.')]
drop_cols.extend(idle_cols)
rate_cols = ['fwd_pkts_per_sec', 'bwd_pkts_per_sec', 'flow_pkts_per_sec', 'payload_bytes_per_second']
rate_cols = [col for col in rate_cols if col in df.columns]
drop_cols.extend(rate_cols)
if 'Attack_type' in df.columns:
    drop_cols.append('Attack_type')

df_reg = df.drop(columns=drop_cols)
print(f"Features after selection: {len(df_reg.columns)}")

In [ ]:
# Create engineered feature
df_reg['total_packets'] = df_reg['fwd_pkts_tot'] + df_reg['bwd_pkts_tot']
print("Created feature: total_packets")

In [ ]:
# Split using saved indices
X = df_reg.drop(columns=[target])
y = df_reg[target]

X_train = X.loc[train_indices]
X_test = X.loc[test_indices]
y_train = y.loc[train_indices]
y_test = y.loc[test_indices]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# Identify categorical and numerical columns
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical features: {cat_cols}")
print(f"Numerical features: {len(num_cols)}")

## 1. Random Forest Regressor

Random Forest combines many decision trees. Each tree learns from different samples and features. The final regression prediction is based on the ensemble of trees. It can capture nonlinear relationships.

In [ ]:
# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ],
    remainder='passthrough'
)

# Create Random Forest pipeline
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42, n_estimators=100))
])

print("Random Forest pipeline created")

In [ ]:
# Train Random Forest
rf_pipeline.fit(X_train, y_train)
print("Random Forest training complete")

In [ ]:
# Predict and evaluate
y_pred_rf = rf_pipeline.predict(X_test)

r2_rf = r2_score(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)

print("Random Forest Results:")
print(f"R²: {r2_rf:.4f}")
print(f"RMSE: {rmse_rf:.4f}")
print(f"MAE: {mae_rf:.4f}")

In [ ]:
# Store results
results = []
results.append({
    'Model': 'Random Forest',
    'R2': r2_rf,
    'RMSE': rmse_rf,
    'MAE': mae_rf
})

results_df = pd.DataFrame(results)
print(results_df)

### Random Forest Feature Importance

Extract and display feature importances from the Random Forest model.

In [ ]:
# Get feature names after preprocessing
feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()
print(f"Total features after encoding: {len(feature_names)}")

In [ ]:
# Get feature importances
importances = rf_pipeline.named_steps['regressor'].feature_importances_

# Create DataFrame for feature importances
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

# Display top features
print("Top 20 most important features:")
print(feature_importance_df.head(20))

## 2. Gradient Boosting Regressor

Gradient Boosting builds trees sequentially. Each new tree tries to improve the errors made by previous trees. It can model nonlinear relationships.

In [ ]:
# Create Gradient Boosting pipeline
gb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(random_state=42, n_estimators=100))
])

print("Gradient Boosting pipeline created")

In [ ]:
# Train Gradient Boosting
gb_pipeline.fit(X_train, y_train)
print("Gradient Boosting training complete")

In [ ]:
# Predict and evaluate
y_pred_gb = gb_pipeline.predict(X_test)

r2_gb = r2_score(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
mae_gb = mean_absolute_error(y_test, y_pred_gb)

print("Gradient Boosting Results:")
print(f"R²: {r2_gb:.4f}")
print(f"RMSE: {rmse_gb:.4f}")
print(f"MAE: {mae_gb:.4f}")

In [ ]:
# Add Gradient Boosting results to the table
results.append({
    'Model': 'Gradient Boosting',
    'R2': r2_gb,
    'RMSE': rmse_gb,
    'MAE': mae_gb
})

results_df = pd.DataFrame(results)
print("Model Comparison:")
print(results_df)

## 3. Support Vector Regression (SVR)

SVR tries to fit a function within an allowed error margin. It is sensitive to feature scale. Kernel functions allow it to model nonlinear relationships.

In [ ]:
# Create SVR pipeline with scaling
# SVR requires feature scaling
# Using linear kernel for computational efficiency on large dataset
# Using a subset of training data for SVR due to computational constraints
svr_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ]
)

svr_pipeline = Pipeline([
    ('preprocessor', svr_preprocessor),
    ('regressor', SVR(kernel='linear', C=1.0))
])

# Use subset of training data for SVR (10% sample)
svr_train_size = int(len(X_train) * 0.1)
X_train_svr = X_train.iloc[:svr_train_size]
y_train_svr = y_train.iloc[:svr_train_size]

print(f"Training SVR on {len(X_train_svr)} samples (subset for computational efficiency)")

In [ ]:
# Train SVR
svr_pipeline.fit(X_train_svr, y_train_svr)
print("SVR training complete")

In [ ]:
# Predict and evaluate on full test set
y_pred_svr = svr_pipeline.predict(X_test)

r2_svr = r2_score(y_test, y_pred_svr)
rmse_svr = np.sqrt(mean_squared_error(y_test, y_pred_svr))
mae_svr = mean_absolute_error(y_test, y_pred_svr)

print("SVR Results:")
print(f"R²: {r2_svr:.4f}")
print(f"RMSE: {rmse_svr:.4f}")
print(f"MAE: {mae_svr:.4f}")

In [ ]:
# Add SVR results to the table
results.append({
    'Model': 'SVR (linear, subset)',
    'R2': r2_svr,
    'RMSE': rmse_svr,
    'MAE': mae_svr
})

results_df = pd.DataFrame(results)
print("Model Comparison:")
print(results_df)

## 4. Lasso Regression

Lasso Regression adds an L1 penalty on the coefficients, which can shrink some coefficients exactly to zero, performing feature selection. The regularization strength is controlled by `alpha`.

In [ ]:
# Create preprocessing pipeline (categorical encoding + numerical scaling)
preprocessor_scaled = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols)
    ]
)

# Create Lasso pipeline
lasso_pipeline = Pipeline([
    ('preprocessor', preprocessor_scaled),
    ('regressor', Lasso(max_iter=5000))
])

print("Lasso pipeline created")

In [ ]:
# Grid of alpha values to compare
param_grid_lasso = {'regressor__alpha': [0.001, 0.01, 0.1, 1.0, 10.0]}

grid_search_lasso = GridSearchCV(
    lasso_pipeline,
    param_grid_lasso,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

print("Lasso GridSearchCV pipeline created")

In [ ]:
# Train with 5-fold cross-validation over the alpha grid
grid_search_lasso.fit(X_train, y_train)

best_alpha_lasso = grid_search_lasso.best_params_['regressor__alpha']
print(f"Best alpha found: {best_alpha_lasso}")
print(f"Best cross-validated R2 (train folds): {grid_search_lasso.best_score_:.4f}")

In [ ]:
# Evaluate the best Lasso model on the held-out test set
best_lasso_pipeline = grid_search_lasso.best_estimator_
y_pred_lasso = best_lasso_pipeline.predict(X_test)

r2_lasso = r2_score(y_test, y_pred_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)

print(f"Lasso Results (best alpha = {best_alpha_lasso}):")
print(f"R²: {r2_lasso:.4f}")
print(f"RMSE: {rmse_lasso:.4f}")
print(f"MAE: {mae_lasso:.4f}")

In [ ]:
# Add to results table
results.append({
    'Model': f'Lasso (alpha={best_alpha_lasso})',
    'R2': r2_lasso,
    'RMSE': rmse_lasso,
    'MAE': mae_lasso
})

results_df = pd.DataFrame(results)
print(results_df)

## 5. Linear Regression

Linear Regression is the baseline model for this track. It assumes a linear relationship between the network-flow features and flow_duration, fitting a coefficient to each feature. Unlike the tree-based models, linear models need scaled numerical features.

In [ ]:
# Create Linear Regression pipeline (same preprocessing as Lasso)
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor_scaled),
    ('regressor', LinearRegression())
])

print("Linear Regression pipeline created")

In [ ]:
# Train Linear Regression
lr_pipeline.fit(X_train, y_train)
print("Linear Regression training complete")

In [ ]:
# Predict and evaluate
y_pred_lr = lr_pipeline.predict(X_test)

r2_lr = r2_score(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr = mean_absolute_error(y_test, y_pred_lr)

print("Linear Regression Results:")
print(f"R²: {r2_lr:.4f}")
print(f"RMSE: {rmse_lr:.4f}")
print(f"MAE: {mae_lr:.4f}")

In [ ]:
# Add to results table
results.append({
    'Model': 'Linear Regression',
    'R2': r2_lr,
    'RMSE': rmse_lr,
    'MAE': mae_lr
})

results_df = pd.DataFrame(results)
print(results_df)

## 6. Ridge Regression

Ridge Regression adds an L2 penalty on the coefficients, which shrinks them toward zero (without setting them exactly to zero) and stabilizes the fit when features are correlated. The regularization strength is controlled by `alpha`.

In [ ]:
# Ridge pipeline (same categorical/numerical preprocessing as Linear Regression)
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor_scaled),
    ('regressor', Ridge())
])

# Grid of alpha values to compare
param_grid_ridge = {'regressor__alpha': [0.01, 0.1, 1.0, 10.0, 50.0, 100.0]}

grid_search_ridge = GridSearchCV(
    ridge_pipeline,
    param_grid_ridge,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

print("Ridge GridSearchCV pipeline created")

In [ ]:
# Train with 5-fold cross-validation over the alpha grid
grid_search_ridge.fit(X_train, y_train)

best_alpha_ridge = grid_search_ridge.best_params_['regressor__alpha']
print(f"Best alpha found: {best_alpha_ridge}")
print(f"Best cross-validated R2 (train folds): {grid_search_ridge.best_score_:.4f}")

In [ ]:
# Evaluate the best Ridge model on the held-out test set
best_ridge_pipeline = grid_search_ridge.best_estimator_
y_pred_ridge = best_ridge_pipeline.predict(X_test)

r2_ridge = r2_score(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)

print(f"Ridge Regression Results (best alpha = {best_alpha_ridge}):")
print(f"R²: {r2_ridge:.4f}")
print(f"RMSE: {rmse_ridge:.4f}")
print(f"MAE: {mae_ridge:.4f}")

In [ ]:
# Add to results table
results.append({
    'Model': f'Ridge (alpha={best_alpha_ridge})',
    'R2': r2_ridge,
    'RMSE': rmse_ridge,
    'MAE': mae_ridge
})

results_df = pd.DataFrame(results)
print(results_df)

## 7. ElasticNet Regression

ElasticNet combines both penalties: an L1 term (like Lasso) and an L2 term (like Ridge). Two hyperparameters control this: `alpha` (overall regularization strength) and `l1_ratio` (the mix between L1 and L2).

In [ ]:
# ElasticNet pipeline (same preprocessing as Linear/Ridge)
elasticnet_pipeline = Pipeline([
    ('preprocessor', preprocessor_scaled),
    ('regressor', ElasticNet(max_iter=5000))
])

# Grid of alpha AND l1_ratio values to tune together
param_grid_en = {
    'regressor__alpha': [0.01, 0.1, 1.0, 10.0],
    'regressor__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

grid_search_en = GridSearchCV(
    elasticnet_pipeline,
    param_grid_en,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

print("ElasticNet GridSearchCV pipeline created")

In [ ]:
# Train with 5-fold cross-validation over the alpha/l1_ratio grid
grid_search_en.fit(X_train, y_train)

best_alpha_en = grid_search_en.best_params_['regressor__alpha']
best_l1_ratio_en = grid_search_en.best_params_['regressor__l1_ratio']

print(f"Best alpha found: {best_alpha_en}")
print(f"Best l1_ratio found: {best_l1_ratio_en}")
print(f"Best cross-validated R2 (train folds): {grid_search_en.best_score_:.4f}")

In [ ]:
# Evaluate the best ElasticNet model on the held-out test set
best_en_pipeline = grid_search_en.best_estimator_
y_pred_en = best_en_pipeline.predict(X_test)

r2_en = r2_score(y_test, y_pred_en)
rmse_en = np.sqrt(mean_squared_error(y_test, y_pred_en))
mae_en = mean_absolute_error(y_test, y_pred_en)

print(f"ElasticNet Results (best alpha={best_alpha_en}, best l1_ratio={best_l1_ratio_en}):")
print(f"R²: {r2_en:.4f}")
print(f"RMSE: {rmse_en:.4f}")
print(f"MAE: {mae_en:.4f}")

In [ ]:
# Add to results table
results.append({
    'Model': f'ElasticNet (alpha={best_alpha_en}, l1_ratio={best_l1_ratio_en})',
    'R2': r2_en,
    'RMSE': rmse_en,
    'MAE': mae_en
})

results_df = pd.DataFrame(results)
print(results_df)

## 8. Decision Tree Regressor

Decision Tree Regressor learns a set of nonlinear if/else splitting rules directly on the network-flow features to predict `flow_duration`. Trees don't require feature scaling. The key hyperparameter is `max_depth`.

In [ ]:
# Decision Trees don't need scaling, so use a lighter preprocessor (encoding only)
dt_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

dt_pipeline = Pipeline([
    ('preprocessor', dt_preprocessor),
    ('regressor', DecisionTreeRegressor(random_state=42))
])

print("Decision Tree pipeline created")

In [ ]:
# Tune max_depth via GridSearchCV with 5-fold cross-validation
param_grid_dt = {'regressor__max_depth': [3, 5, 7, 10, 15, 20, None]}

grid_search_dt = GridSearchCV(
    dt_pipeline,
    param_grid_dt,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

grid_search_dt.fit(X_train, y_train)

best_max_depth = grid_search_dt.best_params_['regressor__max_depth']
print(f"Best max_depth found: {best_max_depth}")
print(f"Best cross-validated R2 (train folds): {grid_search_dt.best_score_:.4f}")

In [ ]:
# Evaluate the best Decision Tree model on the held-out test set
best_dt_pipeline = grid_search_dt.best_estimator_
y_pred_dt = best_dt_pipeline.predict(X_test)

r2_dt = r2_score(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
mae_dt = mean_absolute_error(y_test, y_pred_dt)

print(f"Decision Tree Regressor Results (best max_depth={best_max_depth}):")
print(f"R²: {r2_dt:.4f}")
print(f"RMSE: {rmse_dt:.4f}")
print(f"MAE: {mae_dt:.4f}")

In [ ]:
# Add to results table
results.append({
    'Model': f'Decision Tree (max_depth={best_max_depth})',
    'R2': r2_dt,
    'RMSE': rmse_dt,
    'MAE': mae_dt
})

results_df = pd.DataFrame(results)
print(results_df)

## 9. K-Nearest Neighbors Regressor

KNN Regressor predicts a new flow's `flow_duration` by finding the `k` most similar flows and averaging their durations. Because KNN relies directly on distances, it is sensitive to feature scale. We train KNN both with and without scaling to compare the effect.

In [ ]:
# Pipeline WITH scaling (same preprocessor used for Linear/Ridge/ElasticNet)
knn_scaled_pipeline = Pipeline([
    ('preprocessor', preprocessor_scaled),
    ('regressor', KNeighborsRegressor())
])

# Pipeline WITHOUT scaling (encoding only, numeric features passed through raw)
knn_unscaled_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)
knn_unscaled_pipeline = Pipeline([
    ('preprocessor', knn_unscaled_preprocessor),
    ('regressor', KNeighborsRegressor())
])

print("Scaled and unscaled KNN pipelines created")

In [ ]:
# Tune n_neighbors for the SCALED pipeline via GridSearchCV with 5-fold CV
param_grid_knn = {'regressor__n_neighbors': [3, 5, 7, 10, 15, 20, 30]}

grid_search_knn_scaled = GridSearchCV(
    knn_scaled_pipeline,
    param_grid_knn,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

grid_search_knn_scaled.fit(X_train, y_train)

best_k_scaled = grid_search_knn_scaled.best_params_['regressor__n_neighbors']
print(f"[Scaled] Best n_neighbors found: {best_k_scaled}")
print(f"[Scaled] Best cross-validated R2 (train folds): {grid_search_knn_scaled.best_score_:.4f}")

In [ ]:
# Tune n_neighbors for the UNSCALED pipeline the same way, for a fair comparison
grid_search_knn_unscaled = GridSearchCV(
    knn_unscaled_pipeline,
    param_grid_knn,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

grid_search_knn_unscaled.fit(X_train, y_train)

best_k_unscaled = grid_search_knn_unscaled.best_params_['regressor__n_neighbors']
print(f"[Unscaled] Best n_neighbors found: {best_k_unscaled}")
print(f"[Unscaled] Best cross-validated R2 (train folds): {grid_search_knn_unscaled.best_score_:.4f}")

In [ ]:
# Evaluate the best SCALED KNN model on the held-out test set
best_knn_scaled_pipeline = grid_search_knn_scaled.best_estimator_
y_pred_knn_scaled = best_knn_scaled_pipeline.predict(X_test)

r2_knn_scaled = r2_score(y_test, y_pred_knn_scaled)
rmse_knn_scaled = np.sqrt(mean_squared_error(y_test, y_pred_knn_scaled))
mae_knn_scaled = mean_absolute_error(y_test, y_pred_knn_scaled)

print(f"KNN Regressor (SCALED, k={best_k_scaled}) Results:")
print(f"R²: {r2_knn_scaled:.4f}")
print(f"RMSE: {rmse_knn_scaled:.4f}")
print(f"MAE: {mae_knn_scaled:.4f}")

In [ ]:
# Evaluate the best UNSCALED KNN model on the same held-out test set
best_knn_unscaled_pipeline = grid_search_knn_unscaled.best_estimator_
y_pred_knn_unscaled = best_knn_unscaled_pipeline.predict(X_test)

r2_knn_unscaled = r2_score(y_test, y_pred_knn_unscaled)
rmse_knn_unscaled = np.sqrt(mean_squared_error(y_test, y_pred_knn_unscaled))
mae_knn_unscaled = mean_absolute_error(y_test, y_pred_knn_unscaled)

print(f"KNN Regressor (UNSCALED, k={best_k_unscaled}) Results:")
print(f"R²: {r2_knn_unscaled:.4f}")
print(f"RMSE: {rmse_knn_unscaled:.4f}")
print(f"MAE: {mae_knn_unscaled:.4f}")

In [ ]:
# Add whichever KNN variant genuinely performed better on the test set
if r2_knn_scaled >= r2_knn_unscaled:
    knn_final_label = f'KNN (scaled, k={best_k_scaled})'
    knn_final_r2, knn_final_rmse, knn_final_mae = r2_knn_scaled, rmse_knn_scaled, mae_knn_scaled
else:
    knn_final_label = f'KNN (unscaled, k={best_k_unscaled})'
    knn_final_r2, knn_final_rmse, knn_final_mae = r2_knn_unscaled, rmse_knn_unscaled, mae_knn_unscaled

print(f"Better-performing variant on test set: {knn_final_label}")

results.append({
    'Model': knn_final_label,
    'R2': knn_final_r2,
    'RMSE': knn_final_rmse,
    'MAE': knn_final_mae
})

results_df = pd.DataFrame(results)
print(results_df)

## 10. Polynomial Regression

Polynomial Regression creates polynomial features to capture nonlinear relationships. We use degree 2 for computational efficiency on this large dataset.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Create polynomial features pipeline (degree 2 for efficiency)
poly_pipeline = Pipeline([
    ('preprocessor', preprocessor_scaled),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('regressor', LinearRegression())
])

# Use subset of training data for polynomial regression due to computational constraints
poly_train_size = int(len(X_train) * 0.05)  # 5% subset
X_train_poly = X_train.iloc[:poly_train_size]
y_train_poly = y_train.iloc[:poly_train_size]

print(f"Training Polynomial Regression on {len(X_train_poly)} samples (subset for computational efficiency)")

In [ ]:
# Train Polynomial Regression
poly_pipeline.fit(X_train_poly, y_train_poly)
print("Polynomial Regression training complete")

In [ ]:
# Predict and evaluate on full test set
y_pred_poly = poly_pipeline.predict(X_test)

r2_poly = r2_score(y_test, y_pred_poly)
rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
mae_poly = mean_absolute_error(y_test, y_pred_poly)

print("Polynomial Regression Results (degree=2):")
print(f"R²: {r2_poly:.4f}")
print(f"RMSE: {rmse_poly:.4f}")
print(f"MAE: {mae_poly:.4f}")

In [ ]:
# Add to results table
results.append({
    'Model': 'Polynomial Regression (degree=2)',
    'R2': r2_poly,
    'RMSE': rmse_poly,
    'MAE': mae_poly
})

results_df = pd.DataFrame(results)
print(results_df)

## Final Model Comparison

Compare all 10 regression models on the same held-out test set.

In [ ]:
# Final comparison table
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('R2', ascending=False)

print("=== Final Regression Model Comparison ===")
print(results_df.to_string(index=False))

In [ ]:
# Visualize model performance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# R² comparison
axes[0].barh(results_df['Model'], results_df['R2'], color='steelblue')
axes[0].set_xlabel('R² Score')
axes[0].set_title('Model Comparison: R² Score')
axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.5)

# RMSE comparison
axes[1].barh(results_df['Model'], results_df['RMSE'], color='coral')
axes[1].set_xlabel('RMSE (lower is better)')
axes[1].set_title('Model Comparison: RMSE')

# MAE comparison
axes[2].barh(results_df['Model'], results_df['MAE'], color='lightgreen')
axes[2].set_xlabel('MAE (lower is better)')
axes[2].set_title('Model Comparison: MAE')

plt.tight_layout()
plt.show()

## Key Findings

### Best Performing Models:
1. **Decision Tree Regressor** achieved the highest R² score, demonstrating that tree-based models handle the highly skewed `flow_duration` target effectively.
2. **Random Forest** also performed well, showing the benefit of ensemble methods.

### Linear Model Performance:
- **Linear Regression, Ridge, and Lasso** all performed poorly (negative R²), confirming that the relationship between features and `flow_duration` is highly nonlinear.
- **ElasticNet** performed better than other linear models by combining L1 and L2 regularization, achieving feature sparsity.

### Scaling Effects:
- **KNN Regressor** performed better without scaling, which is counterintuitive but explained by the extreme skew in the target variable.
- This finding highlights that scaling is not universally beneficial for all datasets.

### Computational Considerations:
- **SVR** and **Polynomial Regression** required training on subsets due to computational constraints.
- Tree-based models (Random Forest, Gradient Boosting, Decision Tree) handled the full dataset efficiently.

### Target Skewness Impact:
- The extreme right skew of `flow_duration` (skewness > 100) significantly impacts R² scores.
- MAE remained more stable across models, suggesting it's a more robust metric for this skewed target.

This comprehensive comparison demonstrates the importance of model selection and the impact of data characteristics on model performance.